# Lecture 10: Deep Learning for Text Classification
### NLP Course 2027

---

## Learning Outcomes
- Understand why deep learning outperforms classical ML for text
- Learn LSTM and CNN architectures for text classification
- Implement a text classifier using PyTorch
- Compare deep learning vs classical approaches

**Primary References:** *Practical NLP* Ch.4 | *NLP with Transformers* Ch.1 (background)

## 1. Why Deep Learning for NLP?

### Limitations of Classical ML
```
Classical NLP Pipeline:
  Text → Feature Engineering → ML Model → Prediction
         (manual, domain-specific)

Deep Learning Pipeline:
  Text → Embeddings → Neural Layers → Prediction
         (learned from data, end-to-end)
```

| Property | Classical ML | Deep Learning |
|----------|-------------|---------------|
| Feature engineering | Manual, domain-specific | Automatic |
| Representation | Sparse (TF-IDF) | Dense (embeddings) |
| Context | Limited (n-grams) | Long-range (LSTM, attention) |
| Data needs | Works with less data | Needs more data |
| Interpretability | More transparent | Black box |
| State-of-the-art | Surpassed | Current (with transformers) |

## 2. Neural Network Basics for NLP

```
Input Layer: word embeddings
     ↓
Hidden Layers: learn features
     ↓
Output Layer: class probabilities (softmax)
```

### Key Components
- **Embedding layer**: converts token IDs to dense vectors
- **Recurrent/Convolutional layers**: extract features
- **Dropout**: regularization to prevent overfitting
- **Fully connected layer**: classification head
- **Softmax**: convert scores to probabilities

In [1]:
import torch
import torch.nn as nn
import numpy as np

# Check if GPU available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

Using device: cpu
PyTorch version: 2.2.2


## 3. Recurrent Neural Networks (RNNs)

RNNs process text **sequentially**, maintaining hidden state:

```
  h0 → [RNN] → h1 → [RNN] → h2 → [RNN] → h3 → ... → hT
          ↑            ↑            ↑
         x1           x2           x3
      (word1)      (word2)      (word3)
```

### Vanishing Gradient Problem
For long sequences, gradients become extremely small → earlier words are forgotten.

### LSTM: Long Short-Term Memory
LSTMs solve this with a **cell state** and three gates:

```
     ┌──────────────────────────────────────┐
     │              Cell State (c)           │
     │   ×         +           →             │
     │ forget    input      output            │
     │  gate     gate        gate             │
     └──────────────────────────────────────┘
```
- **Forget gate**: what to forget from cell state
- **Input gate**: what new information to store
- **Output gate**: what to output as hidden state

In [2]:
# Simple LSTM text classifier in PyTorch
class LSTMTextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes,
                 num_layers=1, dropout=0.3, bidirectional=True):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=bidirectional
        )
        self.dropout = nn.Dropout(dropout)
        lstm_output_dim = hidden_dim * 2 if bidirectional else hidden_dim
        self.classifier = nn.Linear(lstm_output_dim, num_classes)

    def forward(self, x):
        # x: (batch_size, seq_len)
        embedded = self.dropout(self.embedding(x))  # (batch, seq, embed)
        lstm_out, (hidden, cell) = self.lstm(embedded)
        # Use last hidden state (concatenate if bidirectional)
        if self.lstm.bidirectional:
            hidden = torch.cat([hidden[-2], hidden[-1]], dim=-1)
        else:
            hidden = hidden[-1]
        hidden = self.dropout(hidden)
        output = self.classifier(hidden)
        return output

# Instantiate model
VOCAB_SIZE = 10000
EMBED_DIM = 100
HIDDEN_DIM = 128
NUM_CLASSES = 2  # binary classification

model = LSTMTextClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, NUM_CLASSES)
print(model)
print(f'\nParameters: {sum(p.numel() for p in model.parameters()):,}')

LSTMTextClassifier(
  (embedding): Embedding(10000, 100, padding_idx=0)
  (lstm): LSTM(100, 128, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (classifier): Linear(in_features=256, out_features=2, bias=True)
)

Parameters: 1,236,034


## 4. Convolutional Neural Networks for Text

TextCNN (Kim, 2014) applies convolutions across word sequences:

```
Sentence: [w1, w2, w3, w4, w5, w6]
           ↓ Embedding
          [e1, e2, e3, e4, e5, e6]    (seq_len × embed_dim)
           ↓ Conv filters (size 2,3,4)
          Multiple feature maps
           ↓ Max-over-time pooling
          Fixed-size representation
           ↓ Dropout → Linear → Softmax
          Class probabilities
```

**Why CNNs work for text:** Filters act as n-gram detectors. A filter of width 3 detects trigram patterns.

In [3]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_filters, filter_sizes,
                 num_classes, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.convolutions = nn.ModuleList([
            nn.Conv1d(in_channels=embed_dim,
                      out_channels=num_filters,
                      kernel_size=fs)
            for fs in filter_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(num_filters * len(filter_sizes), num_classes)

    def forward(self, x):
        # x: (batch, seq_len)
        embedded = self.embedding(x)               # (batch, seq, embed)
        embedded = embedded.permute(0, 2, 1)       # (batch, embed, seq)
        pooled_outputs = []
        for conv in self.convolutions:
            conved = torch.relu(conv(embedded))    # (batch, filters, seq-fs+1)
            pooled = conved.max(dim=2).values      # (batch, filters) max-pooling
            pooled_outputs.append(pooled)
        cat = torch.cat(pooled_outputs, dim=1)     # (batch, filters * n_sizes)
        out = self.classifier(self.dropout(cat))
        return out

# Instantiate TextCNN
cnn_model = TextCNN(
    vocab_size=10000, embed_dim=100,
    num_filters=100, filter_sizes=[2, 3, 4],
    num_classes=2
)
print(cnn_model)
print(f'\nParameters: {sum(p.numel() for p in cnn_model.parameters()):,}')

TextCNN(
  (embedding): Embedding(10000, 100, padding_idx=0)
  (convolutions): ModuleList(
    (0): Conv1d(100, 100, kernel_size=(2,), stride=(1,))
    (1): Conv1d(100, 100, kernel_size=(3,), stride=(1,))
    (2): Conv1d(100, 100, kernel_size=(4,), stride=(1,))
  )
  (dropout): Dropout(p=0.5, inplace=False)
  (classifier): Linear(in_features=300, out_features=2, bias=True)
)

Parameters: 1,090,902


In [4]:
# Complete training pipeline skeleton
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

class TextDataset(Dataset):
    """Simple dataset for tokenized texts."""
    def __init__(self, texts, labels, vocab, max_len=200):
        self.vocab = vocab
        self.max_len = max_len
        self.texts = [self._encode(t) for t in texts]
        self.labels = labels

    def _encode(self, tokens):
        """Convert tokens to integer IDs, pad/truncate."""
        ids = [self.vocab.get(t, 1) for t in tokens[:self.max_len]]  # 1=<UNK>
        # Pad to max_len
        ids += [0] * (self.max_len - len(ids))
        return torch.tensor(ids, dtype=torch.long)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    for texts, labels in dataloader:
        texts, labels = texts.to(device), labels.to(device)
        optimizer.zero_grad()
        output = model(texts)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        preds = output.argmax(dim=1)
        correct += (preds == labels).sum().item()
    return total_loss / len(dataloader), correct / len(dataloader.dataset)

def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    with torch.no_grad():
        for texts, labels in dataloader:
            texts, labels = texts.to(device), labels.to(device)
            output = model(texts)
            loss = criterion(output, labels)
            total_loss += loss.item()
            correct += (output.argmax(dim=1) == labels).sum().item()
    return total_loss / len(dataloader), correct / len(dataloader.dataset)

print('Training skeleton defined.')
print('Usage:')
print('  optimizer = optim.Adam(model.parameters(), lr=1e-3)')
print('  criterion = nn.CrossEntropyLoss()')
print('  for epoch in range(num_epochs):')
print('      train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)')
print('      val_loss, val_acc = evaluate(model, val_loader, criterion, device)')

Training skeleton defined.
Usage:
  optimizer = optim.Adam(model.parameters(), lr=1e-3)
  criterion = nn.CrossEntropyLoss()
  for epoch in range(num_epochs):
      train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
      val_loss, val_acc = evaluate(model, val_loader, criterion, device)


In [5]:
# End-to-end example with NLTK movie reviews
from nltk.corpus import movie_reviews
from sklearn.model_selection import train_test_split
from collections import Counter
import nltk

nltk.download('movie_reviews', quiet=True)

# Load data
texts = [movie_reviews.words(f) for f in movie_reviews.fileids()]
labels_str = [movie_reviews.categories(f)[0] for f in movie_reviews.fileids()]
labels = [1 if l == 'pos' else 0 for l in labels_str]

print(f'Dataset: {len(texts)} documents')
print(f'Positive: {sum(labels)}  Negative: {len(labels)-sum(labels)}')

# Build vocabulary
all_words = [w.lower() for doc in texts for w in doc if w.isalpha()]
counter = Counter(all_words)
VOCAB_SIZE = 10000
vocab = {'<PAD>': 0, '<UNK>': 1}
for word, _ in counter.most_common(VOCAB_SIZE - 2):
    vocab[word] = len(vocab)
print(f'Vocabulary size: {len(vocab)}')

Dataset: 2000 documents
Positive: 1000  Negative: 1000
Vocabulary size: 10000


In [6]:
# Comparison: Classical ML vs Deep Learning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# Prepare text as strings for sklearn
text_strings = [' '.join(doc) for doc in texts]
X_train, X_test, y_train, y_test = train_test_split(
    text_strings, labels, test_size=0.2, random_state=42)

# Classical approach: TF-IDF + Logistic Regression
tfidf = TfidfVectorizer(max_features=10000, stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

lr = LogisticRegression(max_iter=500)
lr.fit(X_train_tfidf, y_train)
lr_acc = accuracy_score(y_test, lr.predict(X_test_tfidf))

print('Approach Comparison on Movie Reviews:')
print('-' * 50)
print(f'Classical (TF-IDF + LR): {lr_acc:.4f}')
print(f'Deep Learning (LSTM):    ~0.86-0.89 (after full training)')
print(f'Fine-tuned BERT:         ~0.94-0.96')
print()
print('Notes:')
print('- Classical ML is fast to train but limited in representation')
print('- LSTM captures sequential context but needs more data/time')
print('- BERT (pre-trained) gives best results with limited fine-tuning')

Approach Comparison on Movie Reviews:
--------------------------------------------------
Classical (TF-IDF + LR): 0.8250
Deep Learning (LSTM):    ~0.86-0.89 (after full training)
Fine-tuned BERT:         ~0.94-0.96

Notes:
- Classical ML is fast to train but limited in representation
- LSTM captures sequential context but needs more data/time
- BERT (pre-trained) gives best results with limited fine-tuning


## 5. Training Tips

| Tip | Why |
|-----|-----|
| Use pretrained embeddings | Better starting point |
| Gradient clipping | Prevents exploding gradients in RNNs |
| Dropout (0.3–0.5) | Regularization |
| Early stopping | Prevent overfitting |
| Learning rate scheduling | Converge faster |
| Batch normalization | Stable training |
| Label smoothing | Calibration |

```python
# Gradient clipping (critical for RNNs)
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3)
```

## Practice Exercises

See **`Lecture-10-Homework.ipynb`** for the practice exercises accompanying this lecture.

## Summary

| Architecture | Strengths | Best For |
|-------------|-----------|----------|
| MLP | Fast, simple | Short texts, BoW features |
| TextCNN | Captures n-gram patterns | Sentence classification |
| LSTM/BiLSTM | Sequential context | Longer text, NER, tagging |
| BiLSTM + Attention | Long-range dependencies | Document classification |
| BERT (next lecture) | Best representations | All NLP tasks |

**Next Lecture**: Transformer Architecture — the self-attention mechanism that powers BERT and GPT.

---
*Book references: Practical NLP Ch.4 | NLP with Transformers Ch.1*

---
**Author: Lei Wu | © 2026 Lei Wu. All rights reserved. Unauthorized use is prohibited.**